# 12. Demo final NPP-Lima: tiempo y magnitud

**Objetivo:** visualizar simultáneamente la probabilidad experimental de que se registre al menos un evento **Mw ≥ 4.5** en los próximos 1, 7 o 30 días, según **NPP Softplus W20** y **Poisson**, y una curva de magnitudes obtenida de la rama `F(m)` del mismo NPP.

**Dos modos:** consulta histórica por fecha y estimación desde hoy. En modo histórico, se informa si hubo eventos registrados posteriormente cuando el catálogo cubre todo el horizonte.

**Advertencias científicas:** los porcentajes temporales no están calibrados para uso operativo. La distribución de magnitudes se normaliza **solo dentro de Mw 4.5 a 8.5** y describe un perfil condicional del siguiente evento objetivo, no la fecha ni la magnitud exacta de un terremoto. La curva de magnitudes no depende del horizonte seleccionado, pues se evalúa al tiempo transcurrido hasta la fecha de consulta.

Archivo complementario obligatorio: `src/npp_magnitude.py`. Mantén sin cambios tu `src/npp_model.py` y los pesos `models/NPP_softplus_W20.weights.h5`.


In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import gradio as gr

raiz = Path.cwd()
if not (raiz / "data").exists():
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "src"))

from utils import datetime_to_hours, truncate_catalog_by_threshold, find_Poisson_MLE
from npp_model import construir_npp
from npp_magnitude import perfil_magnitud

VENTANA = 20
M0PRED = 4.5
MCUT = 3.5
CORTE_HORAS = 450000

ruta_csv = raiz / "data/raw/SISMOS_LIMA_FORMATO_MODELO_cat.csv"
ruta_pesos = raiz / "models/NPP_softplus_W20.weights.h5"
if not ruta_csv.is_file() or not ruta_pesos.is_file():
    raise FileNotFoundError("Faltan el CSV IGP o los pesos Softplus W20 en el repositorio.")

catalogo = pd.read_csv(ruta_csv)
catalogo = datetime_to_hours(catalogo).dropna()
catalogo = truncate_catalog_by_threshold(catalogo, MCUT).reset_index(drop=True)
fechas = pd.to_datetime(catalogo["datetime"]).reset_index(drop=True)
tiempos = catalogo["time"].to_numpy(dtype=float)
magnitudes = catalogo["mw"].to_numpy(dtype=float)

assert fechas.is_monotonic_increasing
assert np.all(np.diff(tiempos) > 0)
assert len(tiempos) > VENTANA + 1

mask_train = tiempos < CORTE_HORAS
T_train = tiempos[mask_train]
M_train = magnitudes[mask_train]

inicio_test = fechas.iloc[np.searchsorted(tiempos, CORTE_HORAS)]
ultimo_registro = fechas.iloc[-1]

mu_t = float(np.log(np.diff(T_train)).mean())
sigma_t = float(np.log(np.diff(T_train)).std())
mu_m = float(np.log(M_train).mean())
sigma_m = float(np.log(M_train).std())

modelo = construir_npp(
    ventana=VENTANA,
    mu_t=mu_t,
    sigma_t=sigma_t,
    mu_m=mu_m,
    sigma_m=sigma_m,
    m0pred=M0PRED,
    activacion_magnitud="softplus",
)
modelo.load_weights(str(ruta_pesos))

tasa_poisson = float(find_Poisson_MLE(T_train, M_train, M0PRED))
fechas_objetivo = fechas[magnitudes >= M0PRED].to_numpy(dtype="datetime64[ns]")

print("Modelo cargado y datos preparados")
print("Fecha inicial disponible para evaluación:", inicio_test)
print("Último registro disponible:", ultimo_registro)
print("Tasa Poisson por hora:", tasa_poisson)



Modelo cargado y datos preparados
Fecha inicial disponible para evaluación: 2012-05-31 06:35:55
Último registro disponible: 2026-11-02 15:28:08
Tasa Poisson por hora: 0.003581180843539231


In [2]:
def tarjeta(nombre, valor, color, subtitulo):
    return f"""
    <div style="flex:1;min-width:240px;border-radius:16px;
        padding:25px 15px;background:{color};color:#fff;text-align:center;">
      <div style="font-size:16px;font-weight:650">{nombre}</div>
      <div style="font-size:clamp(42px,6vw,72px);font-weight:850;
           margin:12px 0;letter-spacing:-2px">{valor*100:.2f}%</div>
      <div style="font-size:13px;opacity:.9">{subtitulo}</div>
    </div>"""


def evaluar_fecha(modo, fecha_elegida, dias):
    """Inferencia temporal NPP/Poisson + magnitudes condicionadas al rango mostrado."""
    hoy_lima = pd.Timestamp(datetime.now(ZoneInfo("America/Lima")).replace(tzinfo=None))

    if modo == "Desde hoy":
        referencia = hoy_lima
    else:
        if not fecha_elegida:
            raise gr.Error("Selecciona una fecha histórica.")
        try:
            referencia = pd.Timestamp(pd.to_datetime(fecha_elegida).date())
        except (TypeError, ValueError):
            raise gr.Error("La fecha histórica no es válida.")
        if referencia < inicio_test:
            raise gr.Error(f"Selecciona una fecha desde {inicio_test:%d/%m/%Y}.")
        if referencia > hoy_lima:
            raise gr.Error("No puedes consultar una fecha histórica futura.")

    dias = int(dias)
    horizonte_horas = dias * 24.0
    indice = np.searchsorted(fechas.to_numpy(dtype="datetime64[ns]"),
                             referencia.to_datetime64(), side="right") - 1
    if indice < VENTANA:
        raise gr.Error("No hay 20 intervalos anteriores para esta fecha.")

    # Exactamente 20 diferencias temporales y magnitudes históricas.
    hist_t = np.diff(tiempos[indice-VENTANA:indice+1]).reshape(1, VENTANA, 1).astype("float32")
    hist_m = magnitudes[indice-VENTANA+1:indice+1].reshape(1, VENTANA, 1).astype("float32")
    ultimo_evento = fechas.iloc[indice]
    transcurridas = (referencia - ultimo_evento).total_seconds() / 3600.0
    if transcurridas < 0:
        raise gr.Error("La referencia precede al evento histórico seleccionado.")

    # Salida temporal acumulada H(t); el historial permanece fijo en este horizonte.
    t_inicial = max(transcurridas, 1e-3)
    t_final = transcurridas + horizonte_horas
    magnitud_auxiliar = np.array([[M0PRED]], dtype="float32")
    def H_en(t_horas):
        dt = np.array([[t_horas]], dtype="float32")
        H, _ = modelo([hist_t, hist_m, dt, magnitud_auxiliar], training=False)
        return float(np.asarray(H)[0, 0])

    delta_H = H_en(t_final) - H_en(t_inicial)
    if not np.isfinite(delta_H) or delta_H < -1e-5:
        raise gr.Error("La rama temporal produjo un incremento de riesgo inválido.")
    p_npp = float(np.clip(-np.expm1(-max(delta_H, 0.0)), 0.0, 1.0))
    p_poisson = float(np.clip(-np.expm1(-tasa_poisson * horizonte_horas), 0.0, 1.0))

    tarjetas = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:10px 0">'
    tarjetas += tarjeta("NPP-LSTM Softplus W20", p_npp, "#173957", "Estimación neuronal no calibrada")
    tarjetas += tarjeta("Poisson", p_poisson, "#116B5B", "Referencia estadística")
    tarjetas += '</div>'

    fin_horizonte = referencia + pd.Timedelta(days=dias)
    if modo == "Histórico" and fin_horizonte <= ultimo_registro:
        inicio = np.searchsorted(fechas_objetivo, referencia.to_datetime64(), side="right")
        fin = np.searchsorted(fechas_objetivo, fin_horizonte.to_datetime64(), side="right")
        cantidad = int(fin-inicio)
        observado = (f"**{cantidad} evento(s) Mw ≥ 4.5** registrado(s) en el horizonte."
                     if cantidad else "**Ningún evento Mw ≥ 4.5 registrado** en el horizonte.")
    else:
        observado = "**No verificable:** el catálogo no cubre totalmente el horizonte consultado."

    alerta_catalogo = ""
    if referencia > ultimo_registro:
        alerta_catalogo = (f"\n\n⚠️ El último registro disponible es del "
                           f"**{ultimo_registro:%d/%m/%Y %H:%M}**. "
                           "Pueden faltar eventos recientes, por lo que esta estimación puede estar desactualizada.")

    texto = f"""### Consulta
**Referencia:** {referencia:%d/%m/%Y %H:%M} · **Horizonte:** {dias} día(s) · **Objetivo:** Mw ≥ 4.5  
**Último evento usado:** {ultimo_evento:%d/%m/%Y %H:%M} · **Horas transcurridas:** {transcurridas:.1f}  
**Resultado observado:** {observado}
{alerta_catalogo}

**Nota:** la regla de clasificación del 50 % es solo ilustrativa; ningún porcentaje garantiza que ocurra un terremoto. La evaluación es retrospectiva/experimental, no una alerta sísmica.
"""

    # Gráfico temporal.
    fig_t, ax_t = plt.subplots(figsize=(7.5, 3.6))
    barras = ax_t.bar(["NPP-LSTM", "Poisson"], [p_npp*100, p_poisson*100],
                      color=["#2467AB", "#15977C"])
    ax_t.set_ylim(0, 110)
    ax_t.set_ylabel("Probabilidad estimada (%)")
    ax_t.set_title(f"Ocurrencia Mw ≥ 4.5 en {dias} día(s)")
    for barra in barras:
        ax_t.text(barra.get_x()+barra.get_width()/2, barra.get_height()+2,
                  f"{barra.get_height():.2f}%", ha="center", fontweight="bold")
    fig_t.tight_layout()
    plt.close(fig_t)

    # Curva de magnitud a partir de F(m), sin reentrenar la red.
    try:
        perfil = perfil_magnitud(modelo, hist_t, hist_m, transcurridas,
                                minimo=4.5, maximo=8.5, puntos=401)
        filas = "  \n".join(
            f"**{nombre}:** {prob*100:.1f}%"
            for nombre, prob in perfil["intervalos"]
        )
        texto_m = f"""### Perfil experimental de magnitudes
**Mediana dentro del rango:** Mw {perfil['mediana_condicional']:.2f}  
**Pico de densidad aproximado:** Mw {perfil['moda_aproximada']:.2f}  
{filas}

**Interpretación:** los porcentajes anteriores suman 100 % **solo dentro de Mw 4.5–8.5**, si ocurre un evento objetivo dentro de ese rango. No son probabilidades de que un sismo ocurra en {dias} días, ni una predicción exacta de su magnitud. La masa sin normalizar en ese rango es {perfil['masa_original_en_rango']:.3f}; si es muy baja, la interpretación es poco estable. La curva se evalúa a la fecha elegida, no depende del horizonte de 1/7/30 días.
"""
        fig_m, ax_m = plt.subplots(figsize=(8.0, 3.9))
        ax_m.plot(perfil["magnitudes"], perfil["densidad_relativa"],
                  color="#3F6BA8", linewidth=2)
        ax_m.fill_between(perfil["magnitudes"], perfil["densidad_relativa"],
                          color="#3F6BA8", alpha=0.14)
        ax_m.axvline(perfil["mediana_condicional"], linestyle="--",
                     color="#D97706", label=f"Mediana Mw {perfil['mediana_condicional']:.2f}")
        ax_m.set_xlabel("Magnitud Mw")
        ax_m.set_ylabel("Densidad relativa en el rango")
        ax_m.set_title("Distribución condicional Mw 4.5–8.5")
        ax_m.legend()
        ax_m.grid(alpha=0.2)
        fig_m.tight_layout()
        plt.close(fig_m)
    except ValueError as exc:
        texto_m = ("### Perfil de magnitud no disponible\n" +
                   f"No se muestra una probabilidad engañosa: {exc}")
        fig_m = None

    return tarjetas, texto, fig_t, texto_m, fig_m


In [3]:
with gr.Blocks(title="NPP-Lima | Tiempo y magnitud") as demo:
    gr.Markdown("""# NPP-Lima 🧠 | Demostración académica
    **Probabilidad experimental de al menos un sismo Mw ≥ 4.5**.  
    **NPP-LSTM Softplus W20 vs. Poisson** | Consulta histórica o desde hoy.  
    ⚠️ Los valores no están calibrados para alertas sísmicas.
    """)

    with gr.Row():
        modo = gr.Radio(["Histórico", "Desde hoy"], value="Histórico", label="Tipo de consulta")
        dias = gr.Radio([1, 7, 30], value=7, label="Horizonte en días")

    fecha = gr.DateTime(value="2026-08-01", include_time=False,
                        type="string", label="Fecha histórica")
    modo.change(fn=lambda m: gr.update(visible=(m == "Histórico")),
                inputs=modo, outputs=fecha)

    boton = gr.Button("Calcular estimación", variant="primary")

    gr.Markdown("## Probabilidad de ocurrencia")
    tarjetas = gr.HTML()
    descripcion = gr.Markdown()
    grafico_t = gr.Plot(label="Comparación temporal")
    gr.Markdown("## Distribución de magnitudes del NPP")
    texto_m = gr.Markdown()
    grafico_m = gr.Plot(label="Curva de magnitudes")

    salida = [tarjetas, descripcion, grafico_t, texto_m, grafico_m]
    boton.click(fn=evaluar_fecha, inputs=[modo, fecha, dias], outputs=salida)
    demo.load(fn=evaluar_fecha, inputs=[modo, fecha, dias], outputs=salida)

demo.launch(inbrowser=True, share=False)


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
